# 08a Lane Geometry Feature Probe v1

이 노트북은 최종 조향 함수를 정하지 않는다. 목적은 07 decoder가 만든 lane 좌표에서 **중앙 수직막대 steering**에 필요한 geometry feature만 안정적으로 뽑는 것이다.

이번 설계의 핵심은 다음과 같다.

```text
기존 target/offset 방식:
  lane 좌표로 aim_x 또는 가상 target line을 만든다.

이번 fixed-base 방식:
  차량 기준 중앙 막대는 고정한다.
  lane은 막대 윗부분을 좌우로 미는 feature만 제공한다.
```

따라서 한쪽 lane에서는 left/right side를 판단하지 않고, offset도 적용하지 않는다. 한쪽 lane은 오직 `local_slope` 방향 cue만 제공한다.

## 1. 이 노트북이 결정하는 변수

08a가 최종적으로 저장하는 geometry 변수는 아래 정도로 제한한다.

| 변수 | 의미 | 사용 위치 |
|---|---|---|
| `mode` | `both`, `single`, `lost` | steering/state machine에서 신뢰도 판단 |
| `local_slope` | lane의 국소 기울기 | 중앙 막대 윗부분을 좌우로 미는 방향 cue |
| `center_error` | both lane 중앙이 화면 중앙에서 벗어난 정도 | both lane일 때만 차선 중앙 복귀에 사용 |
| `lane_center_x` | both lane의 mid 위치 중앙 x | 디버그/시각화 |
| `mean_conf`, `pair_gap_px` | decoder 출력 진단값 | 디버그/필터 튜닝 |

반대로 이번 노트북에서는 아래를 하지 않는다.

```text
- single lane의 side 판단
- lane_x ± offset target 생성
- previous aim_x propagation
- lost recovery steering
- motor 속도 계산
```

In [ ]:
from pathlib import Path
import json
import math

import cv2
import numpy as np
import pandas as pd
from IPython.display import Image, Video, display

PROJECT_ROOT = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization")
EXP_ROOT = PROJECT_ROOT / "10_experiments" / "12_clrkdnet_supervised_rebuild"
REVIEW_ROOT = EXP_ROOT / "review_outputs" / "08a_lane_geometry_probe_v1"
OLD_08_ROOT = EXP_ROOT / "review_outputs" / "08_driving_postprocess_contract_v1"
DECODED_JSONL = OLD_08_ROOT / "tangent_decoded_lanes_field3.jsonl"
FIELD3_ROOT = PROJECT_ROOT / "20_shared_assets" / "dataset" / "lane" / "raw" / "field3"

OVERLAY_DIR = REVIEW_ROOT / "overlays"
VIDEO_DIR = REVIEW_ROOT / "videos"
TABLE_DIR = REVIEW_ROOT / "tables"
CONFIG_DIR = REVIEW_ROOT / "config"
for d in [OVERLAY_DIR, VIDEO_DIR, TABLE_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_W = 1296
RAW_H = 972
CUT_HEIGHT = 445
IMAGE_CENTER_X = RAW_W / 2.0

print("decoded lanes:", DECODED_JSONL)
print("field3 root:", FIELD3_ROOT)
print("review root:", REVIEW_ROOT)

In [ ]:
def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def imread_bgr(path):
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img


def imwrite_bgr(path, img):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ext = path.suffix or ".jpg"
    ok, buf = cv2.imencode(ext, img)
    if not ok:
        raise RuntimeError(f"cv2.imencode failed: {path}")
    buf.tofile(str(path))


def even_sample(items, n):
    items = list(items)
    if len(items) <= n:
        return items
    idx = np.linspace(0, len(items) - 1, n).round().astype(int)
    return [items[i] for i in idx]


def resize_to_width(img, width):
    h, w = img.shape[:2]
    if w == width:
        return img
    scale = width / float(w)
    return cv2.resize(img, (width, int(round(h * scale))), interpolation=cv2.INTER_AREA)


def make_sheet(images, out_path, cols=3, tile_w=720, pad=10, bg=(245, 245, 245)):
    if not images:
        return None
    tiles = [resize_to_width(img, tile_w) for img in images]
    tile_h = max(t.shape[0] for t in tiles)
    rows = int(math.ceil(len(tiles) / cols))
    sheet = np.full((rows * tile_h + (rows + 1) * pad, cols * tile_w + (cols + 1) * pad, 3), bg, dtype=np.uint8)
    for i, tile in enumerate(tiles):
        r, c = divmod(i, cols)
        y0 = pad + r * (tile_h + pad)
        x0 = pad + c * (tile_w + pad)
        sheet[y0:y0 + tile.shape[0], x0:x0 + tile.shape[1]] = tile
    imwrite_bgr(out_path, sheet)
    return out_path

print("helpers ready")

In [ ]:
assert DECODED_JSONL.exists(), DECODED_JSONL
rows = []
decoded_by_key = {}
with DECODED_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        key = obj["key"]
        decoded_by_key[key] = obj["lanes"]
        rel = obj.get("image_rel") or obj.get("rel_path") or obj.get("rel") or obj.get("path")
        if rel is None:
            stem = obj.get("id") or obj.get("key", "").split("::")[-1]
            rel = str(Path("field_drive") / f"{stem}.jpg")
        img_path = FIELD3_ROOT / rel if not Path(rel).is_absolute() else Path(rel)
        rows.append({
            "key": key,
            "image_rel": rel,
            "image_path": str(img_path),
            "decoded_lane_count": len(obj["lanes"]),
        })

records_df = pd.DataFrame(rows)
print("frames:", len(records_df))
display(records_df.groupby("decoded_lane_count").size().to_frame("frames"))
display(records_df.head())

## 2. y 위치와 feature 설정

local fit label 정책에 맞춰 화면 하단부의 국소 geometry만 본다.

```text
top_y    = 0.80 지점  ≈ 866
mid_y    = 0.90 지점  ≈ 918
bottom_y = 1.00 지점  ≈ 971
```

`local_slope`는 `top_y`와 `bottom_y`의 x 차이로 계산한다.

```text
local_slope = (x_top - x_bottom) / (bottom_y - top_y)
```

중앙 수직막대 steering에서는 `mid_y` 위치의 center error와 `local_slope`만 사용하면 된다.

In [ ]:
GEOMETRY_CONFIG = {
    "top_ratio": 0.80,
    "mid_ratio": 0.90,
    "bottom_ratio": 1.00,
    "min_points": 4,
    "min_y_span": 60.0,
    "max_y_distance": 150.0,
    "pair_gap_min_px": 300.0,
    "pair_gap_max_px": 1300.0,
    "single_select": "closest_to_center",
    "preview_slope_px": 260.0,
    "preview_center_px": 260.0,
}

GEOMETRY_CONFIG["top_y"] = CUT_HEIGHT + GEOMETRY_CONFIG["top_ratio"] * (RAW_H - 1 - CUT_HEIGHT)
GEOMETRY_CONFIG["mid_y"] = CUT_HEIGHT + GEOMETRY_CONFIG["mid_ratio"] * (RAW_H - 1 - CUT_HEIGHT)
GEOMETRY_CONFIG["bottom_y"] = CUT_HEIGHT + GEOMETRY_CONFIG["bottom_ratio"] * (RAW_H - 1 - CUT_HEIGHT)

write_json(CONFIG_DIR / "fixed_base_geometry_config_v1.json", GEOMETRY_CONFIG)
display(pd.Series(GEOMETRY_CONFIG).to_frame("value"))
print("saved:", CONFIG_DIR / "fixed_base_geometry_config_v1.json")

In [ ]:
def normalize_lane_points(lane):
    pts = np.array(lane.get("points", []), dtype=np.float32)
    if pts.ndim != 2 or pts.shape[1] != 2:
        return None
    valid = np.isfinite(pts).all(axis=1)
    pts = pts[valid]
    if len(pts) < 2:
        return None
    order = np.argsort(pts[:, 1])
    return pts[order]


def interp_or_nearest_x(points, query_y, max_y_distance):
    ys = points[:, 1]
    xs = points[:, 0]
    if len(points) < 2:
        return np.nan
    y_min, y_max = float(ys.min()), float(ys.max())
    if y_min <= query_y <= y_max:
        return float(np.interp(query_y, ys, xs))
    nearest_idx = int(np.argmin(np.abs(ys - query_y)))
    if abs(float(ys[nearest_idx]) - query_y) <= max_y_distance:
        return float(xs[nearest_idx])
    return np.nan


def lane_feature(lane, cfg, lane_index):
    pts = normalize_lane_points(lane)
    if pts is None or len(pts) < int(cfg["min_points"]):
        return None
    y_span = float(pts[:, 1].max() - pts[:, 1].min())
    if y_span < float(cfg["min_y_span"]):
        return None
    x_top = interp_or_nearest_x(pts, cfg["top_y"], cfg["max_y_distance"])
    x_mid = interp_or_nearest_x(pts, cfg["mid_y"], cfg["max_y_distance"])
    x_bottom = interp_or_nearest_x(pts, cfg["bottom_y"], cfg["max_y_distance"])
    if not np.isfinite([x_top, x_mid, x_bottom]).all():
        return None
    denom = float(cfg["bottom_y"] - cfg["top_y"])
    slope = float((x_top - x_bottom) / denom) if denom > 1e-6 else 0.0
    conf = lane.get("conf", lane.get("score", np.nan))
    return {
        "lane_index": int(lane_index),
        "points": pts,
        "conf": float(conf) if conf is not None and np.isfinite(conf) else np.nan,
        "x_top": float(x_top),
        "x_mid": float(x_mid),
        "x_bottom": float(x_bottom),
        "local_slope": slope,
        "y_span": y_span,
    }


def extract_features(lanes, cfg):
    feats = []
    for i, lane in enumerate(lanes):
        feat = lane_feature(lane, cfg, i)
        if feat is not None:
            feats.append(feat)
    return feats

print("feature helpers ready")

## 3. both / single / lost feature 선택

- `both`: 정상 gap을 가진 lane pair가 있으면 pair 중앙을 사용한다.
- `single`: pair가 없지만 lane feature가 있으면, 화면 중앙에 가장 가까운 feature의 slope만 사용한다.
- `lost`: 사용할 feature가 없다.

중요한 점은 `single`에서 `center_error`를 0으로 둔다는 것이다. 한쪽 lane의 x 위치는 차선 중앙 오차로 해석하지 않는다.

In [ ]:
def best_valid_pair(features, cfg):
    if len(features) < 2:
        return None
    ordered = sorted(features, key=lambda f: f["x_mid"])
    candidates = []
    for i in range(len(ordered)):
        for j in range(i + 1, len(ordered)):
            left, right = ordered[i], ordered[j]
            gap = right["x_mid"] - left["x_mid"]
            if cfg["pair_gap_min_px"] <= gap <= cfg["pair_gap_max_px"]:
                center_x = 0.5 * (left["x_mid"] + right["x_mid"])
                slope = 0.5 * (left["local_slope"] + right["local_slope"])
                confs = [v for v in [left["conf"], right["conf"]] if np.isfinite(v)]
                candidates.append({
                    "left": left,
                    "right": right,
                    "center_x": float(center_x),
                    "local_slope": float(slope),
                    "gap": float(gap),
                    "mean_conf": float(np.mean(confs)) if confs else np.nan,
                    "score": abs(center_x - IMAGE_CENTER_X),
                })
    if not candidates:
        return None
    return min(candidates, key=lambda c: c["score"])


def choose_single_feature(features):
    if not features:
        return None
    return min(features, key=lambda f: abs(f["x_mid"] - IMAGE_CENTER_X))


def geometry_features_for_frame(lanes, cfg):
    features = extract_features(lanes, cfg)
    pair = best_valid_pair(features, cfg)
    if pair is not None:
        center_error = (pair["center_x"] - IMAGE_CENTER_X) / IMAGE_CENTER_X
        return {
            "mode": "both",
            "local_slope": float(pair["local_slope"]),
            "center_error": float(center_error),
            "lane_center_x": float(pair["center_x"]),
            "single_x_mid": np.nan,
            "pair_gap_px": float(pair["gap"]),
            "mean_conf": float(pair["mean_conf"]),
            "selected_lane_indices": [pair["left"]["lane_index"], pair["right"]["lane_index"]],
            "feature_count": len(features),
            "reason": "valid pair: use center_error + averaged local_slope",
        }
    if features:
        feat = choose_single_feature(features)
        return {
            "mode": "single",
            "local_slope": float(feat["local_slope"]),
            "center_error": 0.0,
            "lane_center_x": np.nan,
            "single_x_mid": float(feat["x_mid"]),
            "pair_gap_px": np.nan,
            "mean_conf": float(feat["conf"]) if np.isfinite(feat["conf"]) else np.nan,
            "selected_lane_indices": [feat["lane_index"]],
            "feature_count": len(features),
            "reason": "single feature: use local_slope only, center_error is forced to zero",
        }
    return {
        "mode": "lost",
        "local_slope": np.nan,
        "center_error": 0.0,
        "lane_center_x": np.nan,
        "single_x_mid": np.nan,
        "pair_gap_px": np.nan,
        "mean_conf": 0.0,
        "selected_lane_indices": [],
        "feature_count": 0,
        "reason": "no usable lane feature",
    }

print("geometry feature selector ready")

In [ ]:
feature_rows = []
features_by_key = {}
for row in records_df.itertuples(index=False):
    lanes = decoded_by_key[row.key]
    geom = geometry_features_for_frame(lanes, GEOMETRY_CONFIG)
    out = {
        "key": row.key,
        "image_path": row.image_path,
        "decoded_lane_count": row.decoded_lane_count,
        "mode": geom["mode"],
        "local_slope": geom["local_slope"],
        "center_error": geom["center_error"],
        "lane_center_x": geom["lane_center_x"],
        "single_x_mid": geom["single_x_mid"],
        "pair_gap_px": geom["pair_gap_px"],
        "mean_conf": geom["mean_conf"],
        "selected_lane_indices": json.dumps(geom["selected_lane_indices"]),
        "feature_count": geom["feature_count"],
        "reason": geom["reason"],
    }
    feature_rows.append(out)
    features_by_key[row.key] = geom

features_df = pd.DataFrame(feature_rows)
features_df.to_csv(TABLE_DIR / "field3_fixed_base_geometry_features.csv", index=False, encoding="utf-8-sig")
print("saved:", TABLE_DIR / "field3_fixed_base_geometry_features.csv")
display(features_df.head())
display(features_df.groupby("mode").size().to_frame("frames"))
display(features_df.groupby("mode")[["local_slope", "center_error", "lane_center_x", "pair_gap_px", "mean_conf"]].agg(["count", "mean", "std", "min", "max"]))

In [ ]:
def draw_dashed_line(img, p1, p2, color, thickness=2, dash=12, gap=8):
    p1 = np.array(p1, dtype=np.float32)
    p2 = np.array(p2, dtype=np.float32)
    dist = float(np.linalg.norm(p2 - p1))
    if dist < 1:
        return
    direction = (p2 - p1) / dist
    t = 0.0
    while t < dist:
        start = p1 + direction * t
        end = p1 + direction * min(t + dash, dist)
        cv2.line(img, tuple(start.astype(int)), tuple(end.astype(int)), color, thickness, cv2.LINE_AA)
        t += dash + gap


def draw_lane_points(img, lane, color=(40, 220, 80)):
    pts = normalize_lane_points(lane)
    if pts is None or len(pts) < 2:
        return
    pts_i = pts.astype(np.int32)
    for p in pts_i:
        cv2.circle(img, tuple(p), 3, color, -1, cv2.LINE_AA)
    cv2.polylines(img, [pts_i.reshape(-1, 1, 2)], False, color, 2, cv2.LINE_AA)


def draw_feature_line(img, feat, selected):
    color = (255, 190, 50) if selected else (160, 160, 160)
    thickness = 3 if selected else 1
    p_top = (int(round(feat["x_top"])), int(round(GEOMETRY_CONFIG["top_y"])))
    p_bottom = (int(round(feat["x_bottom"])), int(round(GEOMETRY_CONFIG["bottom_y"])))
    cv2.line(img, p_bottom, p_top, color, thickness, cv2.LINE_AA)
    cv2.circle(img, (int(round(feat["x_mid"])), int(round(GEOMETRY_CONFIG["mid_y"]))), 5, color, -1, cv2.LINE_AA)


def preview_ray_tip(row):
    slope = 0.0 if not np.isfinite(row["local_slope"]) else float(row["local_slope"])
    center_error = float(row["center_error"])
    tip_x = IMAGE_CENTER_X + GEOMETRY_CONFIG["preview_slope_px"] * slope + GEOMETRY_CONFIG["preview_center_px"] * center_error
    return float(np.clip(tip_x, 0, RAW_W - 1))


def overlay_geometry_feature(row, scale_width=960):
    bgr = imread_bgr(row["image_path"])
    out = bgr.copy()
    lanes = decoded_by_key[row["key"]]
    geom = features_by_key[row["key"]]
    selected = set(geom["selected_lane_indices"])
    feats = extract_features(lanes, GEOMETRY_CONFIG)

    for lane in lanes:
        draw_lane_points(out, lane, color=(40, 220, 80))
    for feat in feats:
        draw_feature_line(out, feat, feat["lane_index"] in selected)

    for y, label, color in [
        (GEOMETRY_CONFIG["top_y"], "top_y=0.8", (80, 150, 255)),
        (GEOMETRY_CONFIG["mid_y"], "mid_y=0.9", (80, 80, 255)),
        (GEOMETRY_CONFIG["bottom_y"], "bottom_y=1.0", (80, 150, 255)),
    ]:
        y_i = int(round(y))
        cv2.line(out, (0, y_i), (RAW_W - 1, y_i), color, 1, cv2.LINE_AA)
        cv2.putText(out, label, (18, max(28, y_i - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

    base = (int(round(IMAGE_CENTER_X)), int(round(GEOMETRY_CONFIG["bottom_y"])))
    center_top = (int(round(IMAGE_CENTER_X)), int(round(GEOMETRY_CONFIG["top_y"])))
    cv2.line(out, base, center_top, (230, 230, 230), 3, cv2.LINE_AA)

    if np.isfinite(row["lane_center_x"]):
        cx = int(round(row["lane_center_x"]))
        y_mid = int(round(GEOMETRY_CONFIG["mid_y"]))
        cv2.circle(out, (cx, y_mid), 8, (0, 170, 255), -1, cv2.LINE_AA)
        cv2.line(out, (int(IMAGE_CENTER_X), y_mid), (cx, y_mid), (0, 170, 255), 3, cv2.LINE_AA)

    tip_x = int(round(preview_ray_tip(row)))
    tip = (tip_x, int(round(GEOMETRY_CONFIG["top_y"])))
    cv2.arrowedLine(out, base, tip, (255, 0, 180), 4, cv2.LINE_AA, tipLength=0.22)

    text_lines = [
        f"mode={row['mode']} lanes={row['decoded_lane_count']} features={row['feature_count']}",
        f"slope={row['local_slope']:+.3f} center_error={row['center_error']:+.3f}",
        f"lane_center_x={row['lane_center_x'] if np.isfinite(row['lane_center_x']) else 'nan'} single_x={row['single_x_mid'] if np.isfinite(row['single_x_mid']) else 'nan'}",
        str(row["reason"]),
    ]
    x0, y0 = 18, 34
    for i, txt in enumerate(text_lines):
        y = y0 + i * 28
        cv2.putText(out, txt, (x0 + 2, y + 2), cv2.FONT_HERSHEY_SIMPLEX, 0.72, (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(out, txt, (x0, y), cv2.FONT_HERSHEY_SIMPLEX, 0.72, (245, 245, 245), 2, cv2.LINE_AA)
    return resize_to_width(out, scale_width)

print("overlay helpers ready")

In [ ]:
# 고르게 뽑은 전체 sample
even_rows = even_sample(feature_rows, 12)
even_imgs = [overlay_geometry_feature(r, scale_width=720) for r in even_rows]
even_path = OVERLAY_DIR / "fixed_base_geometry_even_sample_sheet.jpg"
make_sheet(even_imgs, even_path, cols=3, tile_w=720)
print("saved:", even_path)
display(Image(filename=str(even_path)))

# single frame만 모아서 보기
single_rows = [r for r in feature_rows if r["mode"] == "single"]
single_sample = even_sample(single_rows, 12)
single_imgs = [overlay_geometry_feature(r, scale_width=720) for r in single_sample]
single_path = OVERLAY_DIR / "fixed_base_geometry_single_sheet.jpg"
make_sheet(single_imgs, single_path, cols=3, tile_w=720)
print("saved:", single_path)
display(Image(filename=str(single_path)))

# both / lost도 섞어서 보기
mode_sample = []
for mode in ["both", "single", "lost"]:
    mode_rows = [r for r in feature_rows if r["mode"] == mode]
    mode_sample.extend(even_sample(mode_rows, 4))
mode_imgs = [overlay_geometry_feature(r, scale_width=720) for r in mode_sample]
mode_path = OVERLAY_DIR / "fixed_base_geometry_by_mode_sheet.jpg"
make_sheet(mode_imgs, mode_path, cols=3, tile_w=720)
print("saved:", mode_path)
display(Image(filename=str(mode_path)))

## 4. Geometry feature video

영상은 아직 최종 주행이 아니다. 분홍 화살표는 `local_slope`와 `center_error`를 이용한 **preview ray**다.

확인할 것:

- single frame에서 center_error가 0으로 유지되는가?
- slope 방향이 곡선 방향과 맞는가?
- both frame에서 lane center가 중앙으로 잘 계산되는가?
- lost frame에서 ray가 중앙 수직막대로 돌아오는가?

In [ ]:
def write_geometry_video(rows, out_path, fps=12, scale_width=960, max_frames=None):
    out_path = Path(out_path)
    selected = rows if max_frames is None else rows[:max_frames]
    if not selected:
        return None
    first = overlay_geometry_feature(selected[0], scale_width=scale_width)
    h, w = first.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))
    if not writer.isOpened():
        fallback = out_path.with_suffix(".avi")
        writer = cv2.VideoWriter(str(fallback), cv2.VideoWriter_fourcc(*"XVID"), fps, (w, h))
        out_path = fallback
    assert writer.isOpened(), "VideoWriter failed"
    writer.write(first)
    for i, row in enumerate(selected[1:], start=1):
        frame = overlay_geometry_feature(row, scale_width=scale_width)
        if frame.shape[:2] != (h, w):
            frame = cv2.resize(frame, (w, h), interpolation=cv2.INTER_AREA)
        writer.write(frame)
        if i % 300 == 0:
            print(f"video frame {i}/{len(selected)}")
    writer.release()
    return out_path

VIDEO_MAX_FRAMES = None
video_path = write_geometry_video(
    feature_rows,
    VIDEO_DIR / "fixed_base_geometry_field3_full.mp4",
    fps=12,
    scale_width=960,
    max_frames=VIDEO_MAX_FRAMES,
)
print("saved:", video_path)
display(Video(str(video_path), embed=False))

In [ ]:
summary = {
    "name": "08a_fixed_base_geometry_feature_probe_v1",
    "description": "Geometry-only probe for fixed-base vertical-ray steering. Single-lane frames provide slope only; no side classification and no lane_x offset target.",
    "frames": int(len(features_df)),
    "config": GEOMETRY_CONFIG,
    "mode_counts": {str(k): int(v) for k, v in features_df["mode"].value_counts().to_dict().items()},
    "mode_ratios": {str(k): float(v) for k, v in (features_df["mode"].value_counts() / len(features_df)).to_dict().items()},
    "local_slope_summary": features_df["local_slope"].describe().to_dict(),
    "center_error_summary": features_df["center_error"].describe().to_dict(),
    "outputs": {
        "features_csv": str(TABLE_DIR / "field3_fixed_base_geometry_features.csv"),
        "even_sheet": str(even_path),
        "single_sheet": str(single_path),
        "mode_sheet": str(mode_path),
        "video": str(video_path),
    },
}
write_json(REVIEW_ROOT / "fixed_base_geometry_probe_summary_v1.json", summary)
print("saved:", REVIEW_ROOT / "fixed_base_geometry_probe_summary_v1.json")
print(json.dumps({"mode_counts": summary["mode_counts"], "mode_ratios": summary["mode_ratios"]}, indent=2, ensure_ascii=False))

## 5. 다음 단계

08a가 괜찮다면, 다음 단계는 08b에서 steering만 따로 실험한다.

```text
08a output:
  mode, local_slope, center_error

08b steering:
  fixed base point = 화면 중앙 하단
  ray_tip_x = center_x + slope_weight * local_slope + center_weight * center_error
  steer_norm = ray_tip_x를 motor command로 변환
```

이렇게 분리하면 single lane의 side 문제, offset 문제, aim_x drift 문제를 steering 설계에서 덜 끌고 가게 된다.